### # Automatic Label Generation for YOLO Training
This notebook generates bounding box annotations automatically using classical computer vision techniques.

## Combine Datasets
Merge Dataset1 and Dataset2 to create a larger and more diverse dataset.

In [7]:
import os
import shutil

source1 = "Data/raw/Dataset1"
source2 = "Data/raw/Dataset2"
combined = "combined_dataset"

os.makedirs(combined, exist_ok=True)

for folder in [source1, source2]:
    for file in os.listdir(folder):
        if file.endswith(".jpg"):
            shutil.copy(os.path.join(folder, file), combined)

print("Datasets combined!")

Datasets combined!


## Automatic Label Generation

Bounding boxes are generated automatically using gradient detection, thresholding, and morphological operations. The largest detected contour is assumed to be the barcode and converted into YOLO format.

In [8]:
import cv2
import os

source = "combined_dataset"
label_output = "labels_auto"

os.makedirs(label_output, exist_ok=True)

for file in os.listdir(source):
    if file.endswith(".jpg"):
        path = os.path.join(source, file)
        image = cv2.imread(path)
        
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        
        # 🔥 Gradient (detect barcode lines)
        grad = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=-1)
        grad = cv2.convertScaleAbs(grad)
        
        blur = cv2.GaussianBlur(grad, (5,5), 0)
        
        _, thresh = cv2.threshold(blur, 225, 255, cv2.THRESH_BINARY)
        
        # 🔥 Morphological closing (connect barcode)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (21,7))
        morph = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        
        contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            continue
        
        # Largest contour
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        
        h_img, w_img = image.shape[:2]
        
        x_center = (x + w/2) / w_img
        y_center = (y + h/2) / h_img
        width = w / w_img
        height = h / h_img
        
        with open(os.path.join(label_output, file.replace(".jpg", ".txt")), "w") as f:
            f.write(f"0 {x_center} {y_center} {width} {height}")

print("High-quality labels generated!")

High-quality labels generated!


## Dataset Splitting

The dataset is split into training (80%) and validation (20%) sets.  
Images and labels are organized into separate folders to match the YOLO training format.

In [9]:
import random

dataset_path = "dataset"

for split in ["train", "val"]:
    os.makedirs(f"{dataset_path}/images/{split}", exist_ok=True)
    os.makedirs(f"{dataset_path}/labels/{split}", exist_ok=True)

images = [f for f in os.listdir(source) if f.endswith(".jpg")]
random.shuffle(images)

split_idx = int(0.8 * len(images))
train = images[:split_idx]
val = images[split_idx:]

def copy_data(files, split):
    for img in files:
        label = img.replace(".jpg", ".txt")
        
        shutil.copy(f"{source}/{img}", f"{dataset_path}/images/{split}/{img}")
        shutil.copy(f"{label_output}/{label}", f"{dataset_path}/labels/{split}/{label}")

copy_data(train, "train")
copy_data(val, "val")

print("Dataset ready!")

Dataset ready!


## Dataset configuration

In [10]:
with open("data.yaml", "w") as f:
    f.write("""path: dataset
train: images/train
val: images/val

names:
  0: barcode
""")

## YOLOv8 Model Training

The YOLOv8s model is trained on the prepared dataset using the configuration defined in `data.yaml`.

Training includes data augmentation such as rotation, scaling, and flipping to improve model robustness and generalization. The model is trained for 20 epochs with a batch size of 16 and image size of 640.

In [1]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  
model.train(
    data="data.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    patience=10,
    degrees=10,
    scale=0.5,
    fliplr=0.5
)

New https://pypi.org/project/ultralytics/8.4.32 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.28  Python-3.13.5 torch-2.9.1+cpu CPU (Intel Core 7 240H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=10, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimi

C:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

C:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


MLflow: logging run_id(170404f4173241088e4fc157c403b2fc) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to C:\Users\dell\Barcode Detection & Verification for Quality control\runs\detect\train3
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/20         0G      1.568      2.961      1.708         24        640: 100% ━━━━━━━━━━━━ 19/19 55.6s/it 17:3738.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 24.8s/it 1:15<50.5s
                   all         87         87      0.336      0.529      0.305      0.153

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/20         0G      1.496      1.977      1.647         20        640: 100% ━━━

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000283071CADF0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

## Model Inference

The trained YOLO model is used to detect barcodes in validation images, and the predicted bounding boxes are visualized.

In [5]:
import matplotlib.pyplot as plt
import cv2

results = model("dataset/images/val/")

for r in results:
    img = r.plot()  # draws boxes on image
    
    plt.figure(figsize=(6,6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")


image 1/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009083.jpg: 640x480 (no detections), 394.6ms
image 2/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009104.jpg: 480x640 1 barcode, 165.9ms
image 3/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009114.jpg: 480x640 1 barcode, 96.9ms
image 4/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009115.jpg: 640x480 3 barcodes, 131.0ms
image 5/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009117.jpg: 480x640 2 barcodes, 91.4ms
image 6/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009118.jpg: 480x640 1 barcode, 94.7ms
image 7/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009119.jpg: 480x640 3 barcodes, 89.0ms
image 8

C:\Users\dell\AppData\Local\Temp\ipykernel_23824\483094981.py:9: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(6,6))


In [9]:
from ultralytics import YOLO

# Always load latest trained model
model = YOLO("runs/detect/train3/weights/best.pt")

print("Using model from train3")

Using model from train3


## Best Detection Examples

The following examples show successful barcode detection by the YOLOv8 model.  
Bounding boxes are accurately placed around barcode regions with high confidence. from ultralytics import YOLO

The latest trained model (train3) is used for generating predictions to ensure optimal performance. The prediction results are saved and visualized, showing bounding boxes and confidence scores for detected barcode regions.

In [11]:
import matplotlib.pyplot as plt
import cv2

# Run inference again (safe)
results = model("dataset/images/val/")

# Show BEST detections (first few)
for r in results[:3]:
    img = r.plot()
    
    plt.figure(figsize=(5,5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Successful Detection")
    plt.axis("off")
    results = model("dataset/images/val/", save=True)


image 1/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009083.jpg: 640x480 1 barcode, 305.4ms
image 2/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009104.jpg: 480x640 2 barcodes, 239.9ms
image 3/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009114.jpg: 480x640 1 barcode, 213.8ms
image 4/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009115.jpg: 640x480 1 barcode, 213.9ms
image 5/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009117.jpg: 480x640 1 barcode, 229.8ms
image 6/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009118.jpg: 480x640 1 barcode, 226.2ms
image 7/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009119.jpg: 480x640 2 barcodes, 280.0ms
image 8/87

## Select best & worst images

In [12]:
import os
import cv2
import matplotlib.pyplot as plt

# Run inference (no popup, no saving needed here)
results = model("dataset/images/val/")

best = []
worst = []

# Extract confidence scores
for r in results:
    if len(r.boxes) > 0:
        conf = r.boxes.conf.cpu().numpy().max()  # highest confidence
    else:
        conf = 0  # no detection = worst case
    
    best.append((conf, r))
    worst.append((conf, r))

# Sort
best = sorted(best, key=lambda x: x[0], reverse=True)
worst = sorted(worst, key=lambda x: x[0])

# Select top/bottom
best_images = best[:3]
worst_images = worst[:3]

print("Best confidence scores:", [round(x[0],2) for x in best_images])
print("Worst confidence scores:", [round(x[0],2) for x in worst_images])


image 1/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009083.jpg: 640x480 1 barcode, 490.8ms
image 2/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009104.jpg: 480x640 2 barcodes, 361.2ms
image 3/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009114.jpg: 480x640 1 barcode, 370.2ms
image 4/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009115.jpg: 640x480 1 barcode, 456.4ms
image 5/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009117.jpg: 480x640 1 barcode, 369.8ms
image 6/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009118.jpg: 480x640 1 barcode, 460.7ms
image 7/87 C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\images\val\05102009119.jpg: 480x640 2 barcodes, 344.8ms
image 8/87

In [13]:
for conf, r in best_images:
    img = r.plot()
    
    plt.figure(figsize=(5,5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Best Detection (Conf: {conf:.2f})")
    plt.axis("off")

In [14]:
for conf, r in worst_images:
    img = r.plot()
    
    plt.figure(figsize=(5,5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Failure Case (Conf: {conf:.2f})")
    plt.axis("off")

In [15]:
import os
import cv2

os.makedirs("selected_outputs", exist_ok=True)

# Save best detections
for i, (conf, r) in enumerate(best_images):
    cv2.imwrite(f"selected_outputs/best_{i}.jpg", r.plot())

# Save worst detections
for i, (conf, r) in enumerate(worst_images):
    cv2.imwrite(f"selected_outputs/worst_{i}.jpg", r.plot())

print("Images saved in selected_outputs folder!")

Images saved in selected_outputs folder!


## Selected Detection Results

The best and worst predictions are selected based on confidence scores.

Best detections represent accurate barcode localization, while worst detections highlight limitations of the model.

In [16]:
import cv2
import matplotlib.pyplot as plt
import os

folder = "selected_outputs"

images = sorted(os.listdir(folder))

for img_name in images:
    img = cv2.imread(os.path.join(folder, img_name))
    
    plt.figure(figsize=(5,5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(img_name)
    plt.axis("off")

### Observations

- The model successfully detects barcode regions in most images with high confidence.
- Bounding boxes are well-aligned with barcode structures in successful cases.
- Failure cases occur due to:
  - Complex backgrounds
  - Low image quality
  - Imperfect automatically generated annotations

These observations highlight both the strengths and limitations of the model.

## Model Performance Metrics

The performance of the YOLOv8 model is evaluated using precision, recall, and mean Average Precision (mAP).

In [17]:
metrics = model.val(data="data.yaml")

print("Evaluation Metrics:")
print(metrics)

Ultralytics 8.4.28  Python-3.13.5 torch-2.9.1+cpu CPU (Intel Core 7 240H)
val: Fast image access  (ping: 0.10.0 ms, read: 453.3111.7 MB/s, size: 254.0 KB)
val: Scanning C:\Users\dell\Barcode Detection & Verification for Quality control\dataset\labels\val.cache... 87 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 87/87 8.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 6.8s/it 40.9s8.2sss
                   all         87         87      0.803       0.69      0.718       0.43
Speed: 2.9ms preprocess, 431.8ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to C:\Users\dell\Barcode Detection & Verification for Quality control\runs\detect\val
Evaluation Metrics:
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002837BA428D0>
curves: ['P

## Training Performance Analysis

The training graph shows how the model improves over epochs, including loss reduction and mAP increase.

In [20]:
img = cv2.imread("runs/detect/train3/results.png")

plt.figure(figsize=(8,6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Training Performance (Loss and mAP)")
plt.axis("off")
plt.show()

<Figure size 800x600 with 1 Axes>

<Figure size 800x600 with 1 Axes>

<Figure size 800x600 with 1 Axes>

## Improvement Annotation Refinement

Automatically generated annotations may contain inaccuracies.  
Manual correction of a subset of bounding boxes can significantly improve model performance and reduce validation loss fluctuations.

In [ ]:
from ultralytics import YOLO

# Use a stronger model
model = YOLO("yolov8m.pt")

# Train with improved settings
model.train(
    data="data.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    patience=15,
    
    # Augmentation improvements
    degrees=15,
    scale=0.6,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)